# Synthetic Population Pipeline — Patras

Single-file pipeline. Run top-to-bottom.

**Stages:**
1. Setup
2. Filter population to Patras  ← band-14 gate runs here
3. Build school layer
4. Load spatial data
5. Spatial filtering (residential buildings → districts)
6. Household assignment
7. School assignment
8. Save outputs
9. Validation
10. Visualization

**Prerequisites:**
- `data/synthpop/households.json` — full Achaia synthesis output
- `data/SCHOOLS/sxoleia.csv` — national Greek schools CSV
- `data/greece-260611-free.shp/` — Geofabrik Greece OSM extract (see README)
- `data/PATRAS_GIS_DATA/` — Patras district shapefiles

## 1 — Setup

In [ ]:
import json
import math
import os
import statistics
from collections import Counter
from shapely.affinity import translate
from shapely.geometry import Point, box

import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from folium.plugins import MousePosition
from scipy.stats import chisquare

# Set working directory to project root
if os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Project root:', os.getcwd())

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', None)

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
OSM_FOLDER        = 'data/greece-260611-free.shp'
LANDUSE_FILE      = os.path.join(OSM_FOLDER, 'gis_osm_landuse_a_free_1.shp')
BUILDINGS_FILE    = os.path.join(OSM_FOLDER, 'gis_osm_buildings_a_free_1.shp')
DISTRICTS_FOLDER  = 'data/PATRAS_GIS_DATA/GEITONIES_PATRAS'
SCHOOLS_CSV       = 'data/SCHOOLS/sxoleia.csv'
SCHOOLS_SHP       = 'data/SCHOOLS/patras_schools.shp'
SCHOOLS_SHIFTED   = 'data/SCHOOLS/patras_schools_shifted.shp'
HOUSEHOLDS_SRC    = 'data/synthpop/households.json'
HOUSEHOLDS_PATRAS = 'data/synthpop/patras_households.json'
OUT_BUILDINGS     = 'data/initial_state.geojson'
OUT_SCHOOLS       = 'data/schools.geojson'

# ── Constants ─────────────────────────────────────────────────────────────────
EPSG = 4326
PATRAS_LOCATION_ID = '2423701'

# Greek school name → education level integer
# ΤΕΕ (Τεχνικά Επαγγελματικά Εκπαίδευση) are upper-secondary technical schools
# serving age_group 3 (15–19), so they map to edu_level 3.
EDUCATION_LEVEL_MAP = {
    'νηπιαγωγειο': 0,
    'δημοτικο':    1,
    'γυμνασιο':    2,
    'λυκειο':      3,
    'τεε':         3,
}

# Shift applied to correct GPS offset in the schools CSV
SCHOOLS_LAT_DELTA  =  0.00177
SCHOOLS_LONG_DELTA =  0.00294

# Household capacity by OSM building type
SINGLE_FAMILY = {'house', 'detached', 'terrace', 'bungalow', 'semidetached_house'}
MULTI_FAMILY  = {'apartments', 'residential', 'flat', 'dormitory'}

def building_cap(btype):
    if pd.isna(btype) or str(btype).lower() in ('yes', ''):
        return 3
    t = str(btype).lower()
    if t in SINGLE_FAMILY: return 1
    if t in MULTI_FAMILY:  return 5
    return 2

In [ ]:
# ── Helper functions (inlined from util.py) ───────────────────────────────────

def gpd_load(path, bbox=None):
    gdf = gpd.read_file(path, bbox=bbox)
    return gdf.to_crs(EPSG)


def load_districts():
    shp_files = [
        f for f in os.listdir(DISTRICTS_FOLDER)
        if f.endswith('.shp') and f != 'geitonies_08.shp'
    ]
    gdfs = []
    for shp in shp_files:
        try:
            gdfs.append(gpd.read_file(os.path.join(DISTRICTS_FOLDER, shp), encoding='cp1253'))
        except Exception as e:
            print(f'Warning: could not load {shp}: {e}')
    combined = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True)).to_crs(EPSG)
    combined['DISTRICT_ID'] = range(len(combined))
    return combined

## 2 — Filter population to Patras

In [ ]:
with open(HOUSEHOLDS_SRC, 'r', encoding='utf-8') as f:
    all_hh = json.load(f)

households = [h for h in all_hh if h['location_id'] == PATRAS_LOCATION_ID]

with open(HOUSEHOLDS_PATRAS, 'w', encoding='utf-8') as f:
    json.dump(households, f, indent=2, ensure_ascii=False)

n_hh     = len(households)
n_people = sum(len(h['members']) for h in households)
print(f'Filtered {len(all_hh):,} → {n_hh:,} households  |  {n_people:,} individuals')
print(f'Mean household size: {n_people / n_hh:.4f}')
del all_hh

In [ ]:
# ── GATE: band-14 solo rate ───────────────────────────────────────────────────
# Runs immediately after population load, before any spatial work.
# If synthesis bug is still present, the notebook halts here and no contaminated
# output is written to disk.
members_all = [m for h in households for m in h['members']]
band14_total = sum(1 for m in members_all if m['age_group'] == 14)
band14_solo  = sum(1 for h in households
                   if len(h['members']) == 1 and h['members'][0]['age_group'] == 14)
solo_rate = band14_solo / band14_total if band14_total else float('nan')

print(f'Band 14: {band14_total:,} individuals, {band14_solo:,} solo ({solo_rate:.1%})')
for b in (13, 15):
    tot  = sum(1 for m in members_all if m['age_group'] == b)
    solo = sum(1 for h in households if len(h['members']) == 1 and h['members'][0]['age_group'] == b)
    print(f'  (ref) band {b}: {solo/tot:.1%} solo')

assert solo_rate < 0.99, (
    f'GATE FAILED: band 14 is {solo_rate:.1%} solo. '
    'Synthesis bug not fixed — pipeline halted before any spatial work.'
)
print('\nGate passed. Proceeding to spatial pipeline.')

## 3 — Build school layer

In [ ]:
schools_df = pd.read_csv(SCHOOLS_CSV)

# Keep rows with valid coordinates inside the Patras bounding box
schools_df = schools_df.dropna(subset=['lat', 'long'])
schools_df = schools_df[
    (schools_df['lat']  != 0) & (schools_df['long'] != 0) &
    (schools_df['long'] >= 21.69) & (schools_df['long'] <= 21.77) &
    (schools_df['lat']  >= 38.20) & (schools_df['lat']  <= 38.29)
]

schools_gdf = gpd.GeoDataFrame(
    schools_df,
    geometry=[Point(lon, lat) for lon, lat in zip(schools_df['long'], schools_df['lat'])],
    crs='EPSG:4326'
)

if 'onoma' in schools_gdf.columns:
    schools_gdf = schools_gdf.rename(columns={'onoma': 'name'})

# Classify by education level using Greek keyword match
schools_gdf['edu_level'] = None
for keyword, level in EDUCATION_LEVEL_MAP.items():
    mask = schools_gdf['name'].str.contains(keyword, case=False, na=False)
    schools_gdf.loc[mask, 'edu_level'] = level

keep = [c for c in ['name', 'lat', 'long', 'edu_level', 'geometry'] if c in schools_gdf.columns]
schools_gdf = schools_gdf[keep]
schools_gdf.to_file(SCHOOLS_SHP, driver='ESRI Shapefile', encoding='utf-8')

print(f'Schools found: {len(schools_gdf)}')
for lvl, label in enumerate(['νηπιαγωγείο', 'δημοτικό', 'γυμνάσιο', 'λύκειο']):
    n = (schools_gdf['edu_level'] == lvl).sum()
    flag = ' *** ZERO — will break school assignment for this age band ***' if n == 0 else ''
    print(f'  level {lvl} ({label}): {n}{flag}')

In [ ]:
# Apply GPS correction and save shifted shapefile
schools_gdf = gpd.read_file(SCHOOLS_SHP).to_crs('EPSG:4326')
schools_gdf['geometry'] = schools_gdf['geometry'].apply(
    lambda g: translate(g, xoff=SCHOOLS_LONG_DELTA, yoff=SCHOOLS_LAT_DELTA)
)
schools_gdf['school_id'] = range(len(schools_gdf))
# Do not add a second geometry column — to_file rejects GeoDataFrames with >1 geometry column.
schools_gdf.to_file(SCHOOLS_SHIFTED)
print(f'Saved {SCHOOLS_SHIFTED}')

## 4 — Load spatial data

In [ ]:
districts = load_districts()
minx, miny, maxx, maxy = districts.total_bounds
bbox = box(minx, miny, maxx, maxy)

print(f'Districts loaded: {len(districts)}')
print(f'Total census population (2021): {districts["pop2021"].sum():,}')

landuse   = gpd_load(LANDUSE_FILE,   bbox=bbox).cx[minx:maxx, miny:maxy]
buildings = gpd_load(BUILDINGS_FILE, bbox=bbox).cx[minx:maxx, miny:maxy]
buildings = buildings.drop(columns=['name', 'code'], errors='ignore')

schools = gpd_load(SCHOOLS_SHIFTED, bbox=bbox)
# edu_level serialises as string in shapefiles; cast back to int for comparisons
schools['edu_level'] = pd.to_numeric(schools['edu_level'], errors='coerce')

print(f'OSM buildings loaded: {len(buildings):,}')
print(f'OSM landuse patches loaded: {len(landuse):,}')
print(f'Schools loaded: {len(schools)}')
for lvl, label in enumerate(['νηπιαγωγείο', 'δημοτικό', 'γυμνάσιο', 'λύκειο']):
    n = (schools['edu_level'] == lvl).sum()
    flag = ' *** ZERO ***' if n == 0 else ''
    print(f'  level {lvl} ({label}): {n}{flag}')

## 5 — Spatial filtering

In [ ]:
# Keep only residential/retail landuse zones
residential_landuse = (
    landuse[landuse['fclass'].isin(['residential', 'retail'])]
    .drop(columns=['name'], errors='ignore')
)

# Buildings that fall within residential zones
residential_buildings = gpd.sjoin(
    buildings, residential_landuse[['geometry']],
    predicate='within', how='inner'
).drop(columns=['index_right'])

# Assign each building to its district
eligible_buildings = gpd.sjoin(
    residential_buildings,
    districts[['DISTRICT_ID', 'geometry', 'pop2021']],
    predicate='intersects', how='left'
).reset_index(drop=True)

# Assign each school to its district
schools_with_districts = gpd.sjoin(
    schools, districts[['DISTRICT_ID', 'geometry']],
    how='left', predicate='intersects'
).drop(columns=['index_right'], errors='ignore')

orphan_buildings = eligible_buildings['DISTRICT_ID'].isna().sum()
print(f'Eligible buildings: {len(eligible_buildings):,}')
print(f'Orphan buildings (no district): {orphan_buildings}')
print()

# Building type coverage
typed = eligible_buildings['type'].dropna()
typed = typed[~typed.str.lower().isin(['yes', ''])]
coverage = len(typed) / len(eligible_buildings)
print(f'Buildings with meaningful OSM type: {len(typed):,} ({coverage:.1%})')
print(f'Falling to default cap=3: {len(eligible_buildings) - len(typed):,} ({1-coverage:.1%})')

## 6 — Household assignment

In [ ]:
np.random.shuffle(households)
total_households = len(households)
avg_hh_size = n_people / total_households
target_pop_sum = districts['pop2021'].sum()

# Confirm this is the urban-core district total (~187,554), not the full Patras
# municipality figure (~215,922). The gap is the ~13% of the population that lives
# outside the OSM residential footprint and is not placed by this pipeline.
print(f'target_pop_sum (urban-core districts): {target_pop_sum:,}')
print(f'n_people (Patras municipality):        {n_people:,}')
print(f'Unplaced (outside OSM footprint):      {n_people - target_pop_sum:,} '
      f'({(n_people - target_pop_sum)/n_people:.1%})')

if n_people < target_pop_sum:
    raise ValueError(f'Population too small: {n_people:,} < target {target_pop_sum:,}')

eligible_buildings['households'] = pd.Series(
    [[] for _ in range(len(eligible_buildings))], dtype=object
)
col_pos = eligible_buildings.columns.get_loc('households')

hh_index = 0
assigned_people = 0

for district_id, district_buildings in eligible_buildings.groupby('DISTRICT_ID'):
    district_target = districts.loc[
        districts['DISTRICT_ID'] == district_id, 'pop2021'
    ].iloc[0]
    district_people = 0
    indices = list(district_buildings.index)
    n_b = len(indices)

    est_hh = math.ceil(district_target / avg_hh_size)
    base = max(1, est_hh // n_b)
    extra = est_hh % n_b

    # First pass: spread base allocation evenly
    for i, pos in enumerate(indices):
        if hh_index >= total_households:
            break
        cap = min(building_cap(eligible_buildings.at[pos, 'type']), 5)
        n = min(base + (1 if i < extra else 0), cap)
        selected = []
        for _ in range(n):
            if hh_index >= total_households:
                break
            hh = households[hh_index]
            selected.append(hh)
            district_people += len(hh['members'])
            assigned_people += len(hh['members'])
            hh_index += 1
        eligible_buildings.iat[pos, col_pos] = selected

    # Second pass: top up if target not reached
    for pos in indices:
        if hh_index >= total_households or district_people >= district_target:
            break
        cap = min(building_cap(eligible_buildings.at[pos, 'type']), 5)
        current = eligible_buildings.iat[pos, col_pos]
        if len(current) < cap:
            hh = households[hh_index]
            current.append(hh)
            eligible_buildings.iat[pos, col_pos] = current
            district_people += len(hh['members'])
            assigned_people += len(hh['members'])
            hh_index += 1

print(f'\nAssigned {assigned_people:,} / {target_pop_sum:,} target people  ({assigned_people/target_pop_sum:.1%})')
print(f'Households used: {hh_index:,} / {total_households:,}')

In [ ]:
# Per-district validity check
district_assigned = (
    eligible_buildings.explode('households').dropna(subset=['households'])
    .groupby('DISTRICT_ID')['households']
    .apply(lambda hhs: sum(len(h['members']) for h in hhs))
)
pop_check = (
    districts[['DISTRICT_ID', 'NAME', 'pop2021']]
    .merge(district_assigned.rename('assigned'), on='DISTRICT_ID', how='left')
    .fillna(0).sort_values('DISTRICT_ID')
)

n_hh_per_building = eligible_buildings['households'].apply(len)
occupied = n_hh_per_building[n_hh_per_building > 0]
print(f'Building occupancy  mean={occupied.mean():.2f}  median={occupied.median():.0f}  max={occupied.max()}')
print()

underfilled = pop_check[pop_check['assigned'] < pop_check['pop2021']]
if underfilled.empty:
    print('All districts at or above census target.')
else:
    print(f'{len(underfilled)} district(s) below target:')
    for _, r in underfilled.iterrows():
        gap = int(r['pop2021'] - r['assigned'])
        avail = len(eligible_buildings[
            (eligible_buildings['DISTRICT_ID'] == r['DISTRICT_ID']) &
            (eligible_buildings['households'].apply(len) == 0)
        ])
        print(f"  District {int(r['DISTRICT_ID'])} ({r['NAME']}): {int(r['assigned'])}/{int(r['pop2021'])} "
              f"(gap {gap}, empty buildings left {avail})")

## 7 — School assignment

In [ ]:
# Mutates member dicts in households in place (adds 'school_id' key).
# Re-run Stage 2 (filter cell) first if re-running this cell in isolation.
# Distance comparison uses EPSG:4326 degrees — ranking is correct but reported
# distances would be in degrees, not metres. Reproject to EPSG:2100 if you
# need to report distance metrics.

AGE_TO_EDU = {0: 0, 1: 1, 2: 2, 3: 3}  # age_group integer → school edu_level

def nearest_school(geom, edu_level):
    candidates = schools_with_districts[schools_with_districts['edu_level'] == edu_level]
    if candidates.empty:
        return None
    distances = candidates['geometry'].apply(lambda s: geom.centroid.distance(s))
    return candidates.at[distances.idxmin(), 'school_id']

# Cache nearest school per building per level (avoids O(members) repeat scans)
print('Pre-computing nearest schools per building...')
school_cache = {
    idx: {lvl: nearest_school(eligible_buildings.at[idx, 'geometry'], lvl) for lvl in range(4)}
    for idx in eligible_buildings.index
}

print('Assigning schools to students...')
for idx, row in eligible_buildings.iterrows():
    for hh in row['households']:
        for member in hh['members']:
            if member['age_group'] in AGE_TO_EDU:
                member['school_id'] = school_cache[idx][AGE_TO_EDU[member['age_group']]]

print('Done.')

## 8 — Save outputs

In [ ]:
save_buildings = eligible_buildings.drop(
    columns=['fclass', 'pop2021', 'index_right'], errors='ignore'
)
for col in ['osm_id', 'DISTRICT_ID']:
    if col in save_buildings.columns:
        save_buildings[col] = save_buildings[col].astype('Int64')

save_buildings.to_file(OUT_BUILDINGS, driver='GeoJSON')
schools_with_districts.to_file(OUT_SCHOOLS, driver='GeoJSON')
print(f'Saved: {OUT_BUILDINGS}  ({len(save_buildings):,} buildings)')
print(f'Saved: {OUT_SCHOOLS}  ({len(schools_with_districts)} schools)')

## 9 — Validation

The band-14 gate ran in Stage 2 and halted the notebook if the synthesis bug was present.
Cells below require ELSTAT reference values. Fill each `None` placeholder before running.
The gate assertion is not repeated here — `members_all` is already in scope from Stage 2.

In [ ]:
# 9a — T3.1: Household-size distribution vs Western Greece (Section 6.3)
# FILL: Western Greece 2021 proportions for sizes [1, 2, 3, 4, 5+], summing to 1.
WG_REF = None  # e.g. [0.31, 0.27, 0.18, 0.16, 0.08]

assert WG_REF is not None, 'Fill WG_REF before running 9a.'
assert abs(sum(WG_REF) - 1.0) < 1e-6

sizes = [len(h['members']) for h in households]
sc = Counter('5+' if s >= 5 else str(s) for s in sizes)
cats = ['1', '2', '3', '4', '5+']
obs = np.array([sc[c] for c in cats], dtype=float)
obs_p = obs / obs.sum()
chi2, p = chisquare(f_obs=obs, f_exp=np.array(WG_REF) * obs.sum())

print(f"{'size':<5}{'n':>10}{'synth%':>9}{'WG%':>8}")
for i, c in enumerate(cats):
    print(f'{c:<5}{int(obs[i]):>10,}{obs_p[i]:>8.2%}{WG_REF[i]:>8.2%}')
print(f'\nMean household size: {n_people/n_hh:.4f}')
print(f'Chi-square = {chi2:.3f},  p = {p:.4f}  (df=4)')
print('NOTE: report whatever p is. Do not assume the old p=0.38 survives the band-14 fix.')

In [ ]:
# 9b — B1: Age structure vs ELSTAT Achaia 2021 (Section 7.2)
# FILL: Achaia 2021 proportions per broad band + median age, summing to 1.
ELSTAT_AGE        = None  # e.g. {'0-14':0.14,'15-29':0.18,'30-44':0.20,'45-64':0.27,'65+':0.21}
ELSTAT_MEDIAN_AGE = None  # e.g. 45.3

assert ELSTAT_AGE is not None, 'Fill ELSTAT_AGE before running 9b.'
assert abs(sum(ELSTAT_AGE.values()) - 1.0) < 1e-6

COLLAPSE = {
    '0-14':  [0, 1, 2],
    '15-29': [3, 4, 5],
    '30-44': [6, 7, 8],
    '45-64': [9, 10, 11, 12],
    '65+':   [13, 14, 15],
}
age_counts = Counter(m['age_group'] for m in members_all)
band_share = {b: sum(age_counts[k] for k in ks) / len(members_all) for b, ks in COLLAPSE.items()}

def midpoint(k): return 80.0 if k == 15 else 5 * k + 2.5
synth_median = statistics.median([midpoint(m['age_group']) for m in members_all])

print(f"{'band':<8}{'synth%':>9}{'ELSTAT%':>9}{'diff':>8}")
for band in COLLAPSE:
    d = band_share[band] - ELSTAT_AGE[band]
    print(f'{band:<8}{band_share[band]:>8.2%}{ELSTAT_AGE[band]:>9.2%}{d:>+8.2%}')
print(f'\nSynthetic median age: {synth_median:.1f}  |  ELSTAT: {ELSTAT_MEDIAN_AGE}')
print('NOTE: median is a band-midpoint approximation from 5-year bins, not exact.')

In [ ]:
# 9c — T3.3: Marginal accuracy vs ELSTAT IPF inputs (Section 6.1)
# FILL: exact marginals used as IPF input (from synthesis codebase).
ELSTAT_GENDER     = None  # {0: p_male, 1: p_female}
ELSTAT_AGE_FULL   = None  # {0:..., ..., 15:...}  16 bands
ELSTAT_EMPLOYMENT = None  # {0:..., ..., 6:...}
ELSTAT_EDUCATION  = None  # {0:..., ..., 13:...}

def marginal(attr):
    c = Counter(m[attr] for m in members_all)
    tot = sum(c.values())
    return {k: v / tot for k, v in c.items()}

def max_err(synth, ref, name):
    assert ref is not None, f'Fill {name} before running 9c.'
    keys = set(synth) | set(ref)
    errs = {k: abs(synth.get(k, 0) - ref.get(k, 0)) for k in keys}
    worst = max(errs, key=errs.get)
    return errs[worst], worst

checks = [
    ('gender',     marginal('gender'),     ELSTAT_GENDER,     'ELSTAT_GENDER'),
    ('age_group',  marginal('age_group'),  ELSTAT_AGE_FULL,   'ELSTAT_AGE_FULL'),
    ('employment', marginal('employment'), ELSTAT_EMPLOYMENT, 'ELSTAT_EMPLOYMENT'),
    ('education',  marginal('education'),  ELSTAT_EDUCATION,  'ELSTAT_EDUCATION'),
]
print(f"{'dimension':<12}{'max abs err':>14}{'at code':>10}")
for label, synth, ref, rname in checks:
    e, k = max_err(synth, ref, rname)
    print(f'{label:<12}{e:>13.2%}{str(k):>10}')
print('\nManuscript claims max 1.73% (age 30–44). Recompute here after band-14 fix.')

In [ ]:
# 9d — B2: Spatial assignment summary (Section 6.6)
n_hh_per_bldg = eligible_buildings['households'].apply(len)
occupied_mask = n_hh_per_bldg > 0
occ_counts = n_hh_per_bldg[occupied_mask]

district_fill_rates = []
for did, group in eligible_buildings.groupby('DISTRICT_ID'):
    n_total  = len(group)
    n_filled = (group['households'].apply(len) > 0).sum()
    district_fill_rates.append(n_filled / n_total if n_total else 0)

# Even-distribution algorithm has no centroid-fallback path — assert it
fallback = sum(
    1 for _, row in eligible_buildings.iterrows()
    if row.get('is_centroid_fallback', False) or row.get('fallback', False)
)
assert fallback == 0, f'Unexpected centroid fallbacks found: {fallback}'

print(f'Buildings total       : {len(eligible_buildings):,}')
print(f'Buildings occupied    : {occupied_mask.sum():,} ({occupied_mask.mean():.1%})')
print(f'Centroid fallback     : 0  (confirmed — remove this category from manuscript Sec 7.3)')
print()
print(f'District fill rate    : min={min(district_fill_rates):.1%}  '
      f'median={statistics.median(district_fill_rates):.1%}  '
      f'max={max(district_fill_rates):.1%}')
print(f'HH per occupied bldg  : mean={occ_counts.mean():.2f}  '
      f'median={occ_counts.median():.0f}  max={occ_counts.max()}')

## 10 — Visualization

In [ ]:
# Map 1: eligible buildings colored by district
import random
random.seed(42)
district_colors = {
    did: '#{:06x}'.format(random.randint(0, 0xFFFFFF))
    for did in eligible_buildings['DISTRICT_ID'].dropna().unique()
}

center = [districts.geometry.centroid.y.mean(), districts.geometry.centroid.x.mean()]
m1 = folium.Map(location=center, zoom_start=14, tiles='cartodbpositron')

folium.GeoJson(
    districts,
    style_function=lambda x: {'fillColor': '#74a9cf', 'color': 'black',
                               'weight': 0.5, 'fillOpacity': 0.4},
    tooltip=folium.GeoJsonTooltip(fields=['NAME', 'DISTRICT_ID', 'pop2021'],
                                  aliases=['District:', 'ID:', 'Population:'])
).add_to(m1)

folium.GeoJson(
    eligible_buildings,
    style_function=lambda f: {
        'fillColor': district_colors.get(f['properties'].get('DISTRICT_ID'), '#cccccc'),
        'color': 'black', 'weight': 0.2, 'fillOpacity': 0.6,
    },
    # fclass is not present after the residential landuse spatial join — use type instead
    tooltip=folium.GeoJsonTooltip(fields=['type', 'DISTRICT_ID'],
                                  aliases=['Type:', 'District:'])
).add_to(m1)

m1.save('map_buildings.html')
print('Saved map_buildings.html')
m1

In [ ]:
# Map 2: final state — buildings with assigned households + schools
m2 = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')

folium.GeoJson(
    districts,
    name='Districts',
    style_function=lambda x: {'fillColor': '#74a9cf', 'color': 'black',
                               'weight': 0.5, 'fillOpacity': 0.4},
    tooltip=folium.GeoJsonTooltip(fields=['NAME'], aliases=['District:'])
).add_to(m2)

folium.GeoJson(
    OUT_BUILDINGS,
    name='Buildings',
    style_function=lambda f: {'fillColor': '#3186cc', 'color': 'black',
                               'weight': 0.2, 'fillOpacity': 0.5},
    tooltip=folium.GeoJsonTooltip(fields=['households'], aliases=['Households:'])
).add_to(m2)

schools_fg = folium.FeatureGroup(name='Schools')
for _, row in schools_with_districts.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='gold', fill=True, fill_color='gold',
        tooltip=folium.Tooltip(f"{row.get('name', '')} — level {row.get('edu_level', '?')}")
    ).add_to(schools_fg)
schools_fg.add_to(m2)

MousePosition().add_to(m2)
folium.LayerControl(collapsed=False).add_to(m2)
m2.save('map.html')
print('Saved map.html')
m2